In [1]:
# 05_judge_evaluation.ipynb -- LLM-as-judge with notch-gap context (graded) injected; faithfulness grounded in the fixed input

# === Shared configuration (self-contained: one code cell per notebook) ===
from __future__ import annotations
import os, sys, json, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Resolve the repository root (the folder that holds both 'results' and 'notebooks').
def _find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "results").is_dir() and (cand / "notebooks").is_dir():
            return cand
    return start.resolve().parent if start.resolve().name == "notebooks" else start.resolve()

ROOT = _find_root(Path.cwd())
DATA, RAW, PROC = ROOT / "data", ROOT / "data" / "raw", ROOT / "data" / "processed"
RESULTS, FIGS, TABS = ROOT / "results", ROOT / "results" / "figures", ROOT / "results" / "tables"
for d in (DATA, RAW, PROC, RESULTS, FIGS, TABS):
    d.mkdir(parents=True, exist_ok=True)

# Load environment variables (.env is git-ignored; never commit secrets).
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except Exception:
    pass
LLM_MODELS = {
    "weak":   os.getenv("LLM_MODEL_WEAK", ""),
    "mid":    os.getenv("LLM_MODEL_MID", ""),
    "strong": os.getenv("LLM_MODEL_STRONG", ""),
}
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

# Plotting convention: seaborn, grayscale, no caption, 600 dpi, saved as PNG and PDF.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(context="paper", style="ticks")
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 600, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.grid": True, "grid.alpha": 0.3, "image.cmap": "Greys",
})

def gray_palette(n: int):
    # Evenly spaced grays (mid -> dark) for categorical series.
    return sns.color_palette("Greys", max(n + 2, 3))[2:n + 2]

def save_fig(fig, name: str):
    # Save a figure as PNG and PDF at 600 dpi; no caption is added here.
    for ext in ("png", "pdf"):
        fig.savefig(FIGS / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

def save_table(df: pd.DataFrame, name: str, index: bool = False):
    # Persist a results table as CSV and (best-effort) LaTeX for the manuscript.
    df.to_csv(TABS / f"{name}.csv", index=index)
    try:
        (TABS / f"{name}.tex").write_text(df.to_latex(index=index))
    except Exception:
        pass

# Dataframe I/O that needs no parquet engine: use Parquet if one is installed, else CSV.
def _parquet_ok():
    for mod in ("pyarrow", "fastparquet"):
        try:
            __import__(mod); return True
        except Exception:
            pass
    return False
PARQUET = _parquet_ok()
def write_df(df: pd.DataFrame, stem: Path):
    # 'stem' is a path without extension; returns the path actually written.
    if PARQUET:
        p = stem.with_suffix(".parquet"); df.to_parquet(p); return p
    p = stem.with_suffix(".csv"); df.to_csv(p, index=False); return p
def df_exists(stem: Path) -> bool:
    return stem.with_suffix(".parquet").exists() or stem.with_suffix(".csv").exists()
def read_df(stem: Path) -> pd.DataFrame:
    pq, cv = stem.with_suffix(".parquet"), stem.with_suffix(".csv")
    if pq.exists(): return pd.read_parquet(pq)
    if cv.exists(): return pd.read_csv(cv)
    raise FileNotFoundError(f"{stem}.parquet/.csv")

# Fine-grained rating scale (low -> high); used to map labels to an ordinal rank.
RATING_SCALE = ["D","C","CC","CCC-","CCC","CCC+","B-","B","B+","BB-","BB","BB+",
                "BBB-","BBB","BBB+","A-","A","A+","AA-","AA","AA+","AAA"]
RANK = {g: i for i, g in enumerate(RATING_SCALE)}
def to_rank(label):
    # Map a rating label to an ordinal rank; returns np.nan if unmatched.
    if not isinstance(label, str):
        return np.nan
    s = label.strip().upper().replace(" ", "")
    return RANK.get(s, np.nan)

# --- 05: score every variant with the LLM judge, telling the judge the NOTCH GAP for each firm so
#         it can grade (not pass/fail) whether the rationale justifies an override; faithfulness
#         stays grounded in the FIXED quantitative input so P3 (factual error) is isolated ---
# A neutral, graded framing is used on purpose: an imperative "must justify / else score low" floors
# every variant (incl. the clean original), destroying discrimination. Here the judge grades how well
# the qualitative justification is present and consistent with the gap, so P0/P5 (justification intact)
# stay high while P2 (removed) and P4 (direction reversed) fall.
src = PROC / "perturbations"
if not OPENAI_API_KEY:
    print("[skip] no OPENAI_API_KEY in .env")
elif not df_exists(src):
    print("[skip] run 04_perturbation first (no perturbations).")
else:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    JUDGE = LLM_MODELS.get("strong") or LLM_MODELS.get("mid") or LLM_MODELS.get("weak")

    # Recover the per-firm notch gap (delta) from the scored corpus as a fallback. case_id was
    # assigned in 03 as the row position of the SAME source (override_subset first, else
    # corpus_scored), so a positional lookup reproduces it when the row has no delta column.
    scored = PROC / "override_subset"
    scored = scored if df_exists(scored) else (PROC / "corpus_scored")
    delta_map = {}
    if df_exists(scored):
        sdf = read_df(scored).reset_index(drop=True)
        if "delta" in sdf.columns:
            delta_map = {i: int(sdf.iloc[i]["delta"]) for i in range(len(sdf))}

    def row_delta(r):
        # Prefer the delta carried on the row (from 03/04); fall back to positional lookup.
        d = r.get("delta")
        try:
            d = int(d)
        except Exception:
            d = None
        if d is None:
            d = delta_map.get(int(r["case_id"]))
        return d

    def notch_note(d):
        if d is None or d == 0:
            return "NOTCH_GAP: not available (treat as a standalone rating)."
        direction = "ABOVE" if d > 0 else "BELOW"
        # Neutral, graded framing (no hard pass/fail imperative, which floors every score).
        return (f"NOTCH_GAP: the agency rating is {abs(d)} notch(es) {direction} what the standalone "
                f"financials imply, so a good rationale should present a qualitative/contextual "
                f"justification whose direction matches this gap.")

    def ask_json(model, system, user, max_tokens=400):
        for extra in (dict(temperature=0.0, max_tokens=max_tokens),
                      dict(max_completion_tokens=max_tokens), dict()):
            try:
                r = client.chat.completions.create(
                    model=model,
                    messages=[{"role": "system", "content": system},
                              {"role": "user", "content": user}], **extra)
                return r.choices[0].message.content
            except Exception:
                continue
        return ""

    def parse(js, keys):
        try:
            d = json.loads(js)
        except Exception:
            d = {}
        return {k: d.get(k, np.nan) for k in keys}

    RUBRIC = ("You evaluate a credit-rating output. The issuer's agency rating differs from its "
              "standalone financials by a stated NOTCH_GAP, so a good rationale presents a "
              "qualitative/contextual justification (parent/group/government support, industry, "
              "governance) whose direction matches the gap. Score each 1-5 (integers): "
              "grade_accuracy (is the grade plausible given the profile and gap); "
              "grade_rationale_consistency (does the reasoning support the stated grade); "
              "qualitative_adequacy (to what extent the rationale presents a coherent qualitative "
              "justification consistent with the notch gap: 5 = clear and consistent with the gap's "
              "direction, 3 = present but generic or partial, 1 = absent or contradicts the gap); "
              "factual_faithfulness (are the factual claims consistent with the financial profile); "
              "explainability (is the reasoning clear and traceable). "
              "Return strict JSON with those five keys.")
    AXES = ("grade_accuracy", "grade_rationale_consistency", "qualitative_adequacy",
            "factual_faithfulness", "explainability")
    # Technical contrast metric: faithfulness against the FIXED quantitative input.
    FAITH = ("Given FINANCIAL_DATA and ANSWER, return JSON {\"faithfulness\": <0-1>} = the fraction "
             "of the answer's factual/quantitative claims that are entailed by FINANCIAL_DATA. "
             "Ignore whether the final grade is wise, and ignore unverifiable qualitative statements.")

    pert = read_df(src)
    out = []
    for _, r in pert.iterrows():
        ans = f"Rating: {r['rating']}\nRationale: {r['rationale']}"
        fin = str(r.get("fin_profile", "")) or "(none provided)"
        note = notch_note(row_delta(r))
        rubric_user = f"FINANCIAL_PROFILE:\n{fin}\n{note}\nOUTPUT:\n{ans}"
        sc = parse(ask_json(JUDGE, RUBRIC, rubric_user), AXES)
        faith = parse(ask_json(JUDGE, FAITH, f"FINANCIAL_DATA:\n{fin}\nANSWER:\n{ans}"),
                      ["faithfulness"]).get("faithfulness", np.nan)
        out.append({**r.to_dict(), **sc, "ragas_faithfulness": faith})
    scores = pd.DataFrame(out)
    write_df(scores, PROC / "judge_scores")
    print("wrote judge scores:", len(scores), "rows")

wrote judge scores: 1200 rows
